In [1]:
# Установка зависимостей в окружение текущего ядра (достаточно выполнить один раз)
%pip install -q ahpy pymcdm "pandas>=2.0" "numpy>=1.24" "scipy>=1.10" "scikit-learn>=1.3"


Note: you may need to restart the kernel to use updated packages.


## Сравнение библиотек: AHP + TOPSIS (ahpy, pymcdm)

**Назначение.** Свести замеры стенда (`e2e-results`, `bundle-analysis.json`) и ручные метрики (GitHub, NPM, обработка ошибок) в одну матрицу решений, получить веса критериев методом AHP и ранжирование альтернатив — TOPSIS.

**Как пользоваться**
1. Выполните первую ячейку (`!pip install …`) или установите зависимости вручную: `pip install -r analysis/requirements-notebook.txt`.
2. Запустите E2E и соберите артефакты: `e2e-results/rpc-{lib}.json` (RPC), `e2e-results/wallet-mock-{lib}.json` (кошелёк через mock; результаты без mock не используйте, если Synpress нерепрезентативен). В корне должен лежать `bundle-analysis.json` (скрипт `npm run run-bundle-analysis`).
3. Скопируйте `analysis/manual_metrics.example.json` → `analysis/manual_metrics.json` и заполните числа (звёзды, загрузки NPM, субъективная оценка ошибок 0…1).
4. В разделе конфигурации задайте список библиотек `LIBRARIES` и при необходимости сопоставление ключей бандла `BUNDLE_KEY_BY_LIB`.
5. Заполните матрицу попарных сравнений критериев AHP (шкала Саати 1–9). При несогласованности (CR > 0.1) скорректируйте суждения.
6. Выполните ячейки сверху вниз. Итог: таблица с весами, нормализованной матрицей, коэффициентами близости TOPSIS и рангами.

**Добавление третьей библиотеки.** Расширьте `LIBRARIES`, добавьте ключ в `BUNDLE_KEY_BY_LIB`, строку в `manual_metrics.json`, прогоните сборку бандла и E2E с тем же шаблоном имён файлов. Логика ноутбука не меняется.

**Метрики ошибок в проекте.** Автоматически считается доля RPC-операций без успешных замеров (`timings` пустой) — как прокси «надёжности» прогона. Субъективную оценку API ошибок (типы, документация) вносите в `error_handling_score`.

**PCA по RPC.** Для многих операций строится матрица «библиотека × операция» (среднее время). PCA сжимает её в 1–2 компоненты; по умолчанию в MCDA можно взять среднее время по всем операциям или первую главную компоненту (настраивается).


In [2]:
from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# --- пути: корень проекта (где bundle-analysis.json) ---
def project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "bundle-analysis.json").exists():
        return cwd
    if (cwd.parent / "bundle-analysis.json").exists():
        return cwd.parent
    return cwd


ROOT = project_root()
E2E_DIR = ROOT / "e2e-results"
FIXTURE_DIR = ROOT / "analysis" / "fixtures"
BUNDLE_JSON = ROOT / "bundle-analysis.json"
MANUAL_JSON = ROOT / "analysis" / "manual_metrics.json"
MANUAL_EXAMPLE = ROOT / "analysis" / "manual_metrics.example.json"

# --- список альтернатив (библиотек). Добавьте третью строку при появлении нового адаптера ---
LIBRARIES: list[str] = ["ethers", "viem"]

# Ключи в bundle-analysis.json по libId (см. vite-конфиги dist-ethers / dist-viem)
BUNDLE_KEY_BY_LIB: dict[str, str] = {
    "ethers": "dist-ethers",
    "viem": "dist-viem",
}

# Использовать для RPC-латентности среднее по операциям ("mean") или первую главную компоненту PCA ("pc1")
RPC_LATENCY_MODE: str = "mean"  # "mean" | "pc1"

def resolve_results_file(filename: str) -> Path:
    # Сначала e2e-results/, иначе analysis/fixtures/
    p = E2E_DIR / filename
    if p.exists():
        return p
    fp = FIXTURE_DIR / filename
    if fp.exists():
        warnings.warn(f"Используется фикстура {fp}", stacklevel=2)
        return fp
    return p


print("ROOT =", ROOT)


ROOT = D:\ITMOMagistracy\VKR\playground_v2


In [3]:
def load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Нет файла: {path}")
    with path.open(encoding="utf-8") as f:
        return json.load(f)


def rpc_operation_means(rpc: dict[str, Any]) -> tuple[dict[str, float], float]:
    # Средние времена по operationId; success_ratio — доля операций с непустым timings
    ops = rpc.get("operations") or []
    means: dict[str, float] = {}
    ok = 0
    for o in ops:
        oid = o.get("operationId", "")
        t = o.get("timings") or []
        st = o.get("stats") or {}
        m = st.get("mean")
        if m is not None and t:
            means[oid] = float(m)
            ok += 1
        else:
            means[oid] = float("nan")
    n = len(ops) if ops else 1
    success_ratio = ok / n
    return means, success_ratio


def load_wallet_mock(path: Path) -> dict[str, float]:
    w = load_json(path)
    wc = w.get("warmConnect") or {}
    sg = w.get("sign") or {}
    sn = w.get("send") or {}
    wc_stats = wc.get("stats") or {}
    sg_stats = sg.get("stats") or {}
    sn_stats = sn.get("stats") or {}
    return {
        "cold_connect_ms": float(w.get("coldConnectMs") or 0),
        "warm_connect_mean_ms": float(wc_stats.get("mean") or 0),
        "sign_mean_ms": float(sg_stats.get("mean") or 0),
        "send_mean_ms": float(sn_stats.get("mean") or 0),
    }


def load_manual(path: Path) -> dict[str, dict[str, Any]]:
    if path.exists():
        data = load_json(path)
        return {k: v for k, v in data.items() if not k.startswith("_")}
    warnings.warn(f"{path} не найден — используем пример или пустые значения", stacklevel=2)
    if MANUAL_EXAMPLE.exists():
        data = load_json(MANUAL_EXAMPLE)
        return {k: v for k, v in data.items() if not k.startswith("_")}
    return {lib: {} for lib in LIBRARIES}


In [ ]:
# --- Сбор таблицы признаков по каждой библиотеке ---

rows: list[dict[str, Any]] = []
all_op_ids: set[str] = set()

for lib in LIBRARIES:
    row: dict[str, Any] = {"lib": lib}

    rpc_path = resolve_results_file(f"rpc-{lib}.json")
    if rpc_path.exists():
        rpc = load_json(rpc_path)
        means, sr = rpc_operation_means(rpc)
        row["rpc_success_ratio"] = sr
        row["_rpc_means"] = means
        all_op_ids.update(means.keys())
    else:
        warnings.warn(f"Нет {rpc_path}", stacklevel=2)
        row["rpc_success_ratio"] = float("nan")
        row["_rpc_means"] = {}

    wm_path = resolve_results_file(f"wallet-mock-{lib}.json")
    if wm_path.exists():
        wm = load_wallet_mock(wm_path)
        row.update(
            {
                "wallet_cold_connect_ms": wm["cold_connect_ms"],
                "wallet_warm_mean_ms": wm["warm_connect_mean_ms"],
                "wallet_sign_mean_ms": wm["sign_mean_ms"],
                "wallet_send_mean_ms": wm["send_mean_ms"],
            }
        )
    else:
        warnings.warn(f"Нет {wm_path} — столбцы кошелька будут NaN", stacklevel=2)
        row.update(
            {
                "wallet_cold_connect_ms": float("nan"),
                "wallet_warm_mean_ms": float("nan"),
                "wallet_sign_mean_ms": float("nan"),
                "wallet_send_mean_ms": float("nan"),
            }
        )

    if BUNDLE_JSON.exists():
        bundle = load_json(BUNDLE_JSON)
        bkey = BUNDLE_KEY_BY_LIB.get(lib)
        if bkey and bkey in bundle:
            row["bundle_gzip"] = float(bundle[bkey]["totalGzip"])
            row["bundle_brotli"] = float(bundle[bkey]["totalBrotli"])
        else:
            row["bundle_gzip"] = float("nan")
            row["bundle_brotli"] = float("nan")
    else:
        row["bundle_gzip"] = float("nan")
        row["bundle_brotli"] = float("nan")

    manual = load_manual(MANUAL_JSON).get(lib, {})
    row["github_stars"] = float(manual.get("github_stars", np.nan))
    row["npm_weekly_downloads"] = float(manual.get("npm_weekly_downloads", np.nan))
    row["error_handling_score"] = float(manual.get("error_handling_score", np.nan))

    rows.append(row)

features = pd.DataFrame(rows)
features


In [ ]:
# --- PCA по RPC: матрица библиотеки × операция (средние ms), затем первая компонента ---

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

op_ids = sorted(all_op_ids)

# Сначала среднее по операциям (игнорируя NaN)
def row_mean_means(r: pd.Series) -> float:
    d = r["_rpc_means"]
    vals = [v for v in d.values() if v is not None and np.isfinite(v)]
    return float(np.mean(vals)) if vals else np.nan


features["rpc_mean_ms"] = features.apply(row_mean_means, axis=1)

M = []
for _, r in features.iterrows():
    means = r["_rpc_means"]
    M.append([means.get(oid, np.nan) for oid in op_ids])

X = np.asarray(M, dtype=float)
if X.size and np.isfinite(X).any():
    imp = SimpleImputer(strategy="mean")
    Xf = imp.fit_transform(X)
    # строки: библиотеки; столбцы: операции
    pca = PCA(n_components=min(2, Xf.shape[1], Xf.shape[0]))
    Z = pca.fit_transform(Xf)
    pc1 = Z[:, 0]
    # Выровнять знак PC1 с rpc_mean_ms (больше задержка → больше «стоимость»)
    if len(features) >= 2 and np.isfinite(features["rpc_mean_ms"]).all():
        c = np.corrcoef(features["rpc_mean_ms"].astype(float), pc1)[0, 1]
        if np.isfinite(c) and c < 0:
            pc1 = -pc1
    features["rpc_pc1"] = pc1
    explained = pca.explained_variance_ratio_
    print("PCA explained variance ratio:", explained)
else:
    features["rpc_pc1"] = np.nan
    print("PCA пропущен: нет данных по RPC")

if RPC_LATENCY_MODE == "pc1":
    features["rpc_latency_for_mcda"] = features["rpc_pc1"]
elif RPC_LATENCY_MODE == "mean":
    features["rpc_latency_for_mcda"] = features["rpc_mean_ms"]
else:
    raise ValueError("RPC_LATENCY_MODE должен быть 'mean' или 'pc1'")

features[["lib", "rpc_mean_ms", "rpc_pc1", "rpc_latency_for_mcda", "rpc_success_ratio"]]


In [ ]:
# --- Ручная нормализация экосистемы: объединение github + npm в один столбец 0..1 (min-max по альтернативам) ---

def minmax_series(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(0.5, index=s.index)
    return (s - lo) / (hi - lo)


gh = minmax_series(features["github_stars"])
npm = minmax_series(features["npm_weekly_downloads"])
features["ecosystem_index"] = 0.5 * gh + 0.5 * npm

# Уберём служебные колонки из финального экспорта
export_cols = [
    "lib",
    "bundle_gzip",
    "rpc_latency_for_mcda",
    "rpc_success_ratio",
    "wallet_warm_mean_ms",
    "wallet_sign_mean_ms",
    "wallet_send_mean_ms",
    "ecosystem_index",
    "error_handling_score",
]
summary = features[[c for c in export_cols if c in features.columns]].copy()
summary


In [ ]:
# --- Матрица решений для TOPSIS: строки — библиотеки, столбцы — критерии ---

CRITERIA_COLS = [
    "bundle_gzip",
    "rpc_latency_for_mcda",
    "rpc_success_ratio",
    "wallet_warm_mean_ms",
    "wallet_sign_mean_ms",
    "wallet_send_mean_ms",
    "ecosystem_index",
    "error_handling_score",
]

# 1 = максимум лучше, -1 = минимум лучше
TYPES = np.array([-1, -1, 1, -1, -1, -1, 1, 1])

X_dec = summary[CRITERIA_COLS].to_numpy(dtype=float)
alternatives = summary["lib"].tolist()
print("Альтернативы:", alternatives)
print("Критерии:", CRITERIA_COLS)


In [ ]:
# --- AHP: веса критериев (заполните попарные сравнения по шкале Саати) ---
# AHPy принимает словарь {(имя_i, имя_j): отношение}, где значение — во сколько раз i важнее j.

import ahpy

n = len(CRITERIA_COLS)

# Верхний треугольник матрицы: M[i,j] = насколько критерий i важнее j (i < j). ЗАМЕНИТЕ на свои суждения.
rough = [
    [1, 1 / 2, 2, 2, 2, 2, 3, 3],
    [2, 1, 3, 3, 3, 3, 4, 4],
    [1 / 2, 1 / 3, 1, 1, 1, 1, 2, 2],
    [1 / 2, 1 / 3, 1, 1, 1, 1, 2, 2],
    [1 / 2, 1 / 3, 1, 1, 1, 1, 2, 2],
    [1 / 2, 1 / 3, 1, 1, 1, 1, 2, 2],
    [1 / 3, 1 / 4, 1 / 2, 1 / 2, 1 / 2, 1 / 2, 1, 1],
    [1 / 3, 1 / 4, 1 / 2, 1 / 2, 1 / 2, 1 / 2, 1, 1],
]
M = np.array(rough, dtype=float)

comparisons: dict[tuple[str, str], float] = {}
for i in range(n):
    for j in range(i + 1, n):
        comparisons[(CRITERIA_COLS[i], CRITERIA_COLS[j])] = float(M[i, j])

cmp = ahpy.Compare(name="criteria", comparisons=comparisons, precision=4, random_index="saaty")
weights_dict = cmp.target_weights
print("CR =", cmp.consistency_ratio)
if cmp.consistency_ratio > 0.1:
    warnings.warn("CR > 0.1 — скорректируйте матрицу AHP", stacklevel=2)

weights = np.array([weights_dict[c] for c in CRITERIA_COLS], dtype=float)
weights = weights / weights.sum()
print("Веса:", dict(zip(CRITERIA_COLS, weights)))


In [ ]:
# --- TOPSIS (pymcdm) ---

from pymcdm.methods import TOPSIS

# Удаляем строки с NaN в критичных ячейках (или импутируйте вручную)
mask = np.isfinite(X_dec).all(axis=1)
if not mask.all():
    warnings.warn("Есть NaN в матрице решений — соответствующие альтернативы отброшены", stacklevel=2)

X_use = X_dec[mask]
alts_use = np.array(alternatives)[mask]
topsis = TOPSIS()
pref = topsis(X_use, weights, TYPES)

# Ранг: 1 — лучший (больше pref)
order = np.argsort(-pref)
rank = np.empty_like(order)
rank[order] = np.arange(1, len(order) + 1)

result = pd.DataFrame({"lib": alts_use, "topsis_closeness": pref, "rank": rank}).sort_values("rank")
result


In [ ]:
# --- Опционально: обратный ранг для отчёта ---

result.set_index("lib")[["topsis_closeness", "rank"]]


### Примечания для текста ВКР

- **Связка AHP + TOPSIS:** AHP задаёт веса критериев с учётом предметной области (frontend Web3); TOPSIS относит альтернативы к идеальному и анти-идеальному решениям в нормализованном пространстве критериев.
- **PCA** позволяет не включать в матрицу десятки отдельных RPC-операций, а заменить их одной латентной переменной; при этом среднее по операциям проще интерпретировать.
- **Экосистема и ошибки** вносятся вручную — это осознанное ограничение; при желании замените `ecosystem_index` на отдельные столбцы NPM и GitHub с дополнительными весами в AHP.
